<a href="https://colab.research.google.com/github/kbng2572/Project-Soul/blob/main/Copy_of_llamacpp_colab_free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building llama.cpp from Scratch on Linux (Colab Edition)

Hi! In this tutorial, we are going to compile `llama.cpp` directly from its source code. `llama.cpp` is a highly optimized C/C++ engine that allows you to run massive AI models locally on your own hardware.

While we are using Google Colab for this guide, **the commands you learn here are 100% transferable to a Linux PC** running Ubuntu or Debian.

Today, we will build the engine, enable GPU acceleration, and run **Gemma-4-26B**, an instruction-tuned model from Unsloth.

### Crucial First Step: Enable the GPU
To compile the engine with GPU support, we need to make sure Colab has assigned us a GPU.
1. Go to the top menu and click **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or any available GPU).
3. Click **Save**.

In [4]:
# Check if the NVIDIA GPU is attached and working
!nvidia-smi

Thu Jul 16 07:05:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Clone the Source Code
Before we can build the software, we need to download the blueprints. We do this by "cloning" the official `llama.cpp` repository from GitHub.

*Note: In Google Colab, we use the `%cd` command instead of the standard `cd`. This is a special Jupyter Notebook command that ensures the directory change applies to all the cells that follow.*

In [5]:
# Download the llama.cpp repository
!git clone https://github.com/ggerganov/llama.cpp

# Move inside the folder we just downloaded
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 103952, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 103952 (delta 99), reused 43 (delta 43), pack-reused 103743 (from 3)
Receiving objects: 100% (103952/103952), 409.49 MiB | 29.97 MiB/s, done.
Resolving deltas: 100% (72637/72637), done.
/content/llama.cpp


## Step 2: Compile with CUDA (GPU Acceleration)

Now we need to compile the C++ source code into an executable program. Because we want our model to run fast, we need to tell the compiler to utilize NVIDIA's CUDA toolkit for GPU acceleration.

We use a build system called **CMake** to configure and build the software.

### 💡 The Secret Flag: `CMAKE_CUDA_ARCHITECTURES`
When compiling with CUDA, we use a special flag called `-DCMAKE_CUDA_ARCHITECTURES`. It is highly recommended to understand how this flag works:

* **Option A: Setting it to `native`** (`-DCMAKE_CUDA_ARCHITECTURES=native`)
  * **What it does:** It tells the compiler to look at the exact NVIDIA GPU currently installed in the machine (e.g., the Tesla T4 in Colab) and optimize the binary *only* for that specific hardware.
  * **Pros:** Compilation is **much faster** (often saving several minutes) because it only builds one version of the code, and it runs with peak efficiency on this specific machine.
  * **Cons:** The resulting executable is hardware-locked. If you download the compiled program and try to run it on a PC with a different GPU (like an RTX 3080 or 4090), it will likely crash with an illegal instruction error.

* **Option B: Omitting the flag (Default)**
  * **What it does:** The compiler will build support for *every* common NVIDIA GPU architecture (Pascal, Volta, Turing, Ampere, Ada Lovelace, etc.) into a single, generic executable.
  * **Pros:** Highly portable. You can compile the program here, download it, and run it on almost any modern NVIDIA Linux PC.
  * **Cons:** Compilation takes significantly longer because the machine is compiling the software multiple times over for architectures it isn't even using.

### Which one should you use?
* Use **`native`** if you are building the software directly on the machine you plan to run it on (like we are doing here in Colab).
* **Omit it** if you are building a universal distribution package meant to be shared across different computers with unknown hardware.

Since we are running this directly inside this Colab environment, we will use `native` to save time!

In [6]:
# 1. Configure the build with CUDA enabled and optimize specifically for this machine's GPU
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native

# 2. Compile the code using all available CPU cores for maximum speed
!cmake --build build --config Release -j $(nproc)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

## Step 3: Download the AI Model
With our engine built, we need an AI to run. We are going to download the **Gemma-4-26B** model in the `GGUF` format.

GGUF is a special file format created specifically for `llama.cpp` that contains everything the model needs (weights, configuration, and vocabulary) in a single file.

We will use the Linux `wget` command to download it directly from Hugging Face. *(Note: If you look at the raw Hugging Face URL, we replaced `/blob/` with `/resolve/` so that `wget` downloads the actual file instead of the webpage).*

In [ ]:
# Download the Gemma-4 GGUF file and save it as "gemma-model.gguf"
!wget -O gemma-model.gguf https://huggingface.co/unsloth/gemma-4-26B-A4B-it-qat-GGUF/resolve/main/gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf

--2026-07-16 08:13:43--  https://huggingface.co/unsloth/gemma-4-26B-A4B-it-qat-GGUF/resolve/main/gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf
Resolving huggingface.co (huggingface.co)... 3.165.160.61, 3.165.160.11, 3.165.160.12, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.61|:443... connected.
HTTP request sent, awaiting response... 504 Gateway Timeout
Retrying.

--2026-07-16 08:14:14--  (try: 2)  https://huggingface.co/unsloth/gemma-4-26B-A4B-it-qat-GGUF/resolve/main/gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 504 Gateway Timeout
Retrying.

--2026-07-16 08:14:46--  (try: 3)  https://huggingface.co/unsloth/gemma-4-26B-A4B-it-qat-GGUF/resolve/main/gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 504 Gateway Timeout
Retrying.

--2026-07-16 08:15:19--  (try: 4)  https://huggingface.co/unsloth/gemma-4-26B-A4B-it-qat

## Step 4: Run the Model!
It's time to talk to the AI. We will use the `llama-cli` tool we compiled in Step 2.

Here is what our command-line arguments mean:
* `-m gemma-model.gguf`: Tells the engine which model file to use.
* `-p "..."`: This is the prompt we are sending to the AI.
* `-n 4000`: Limits the AI's response to 4000 tokens (words/word fragments). you can set it to higher number or remove it all together, in that case the defailt value i.e -1 takes effect, the ai will keep generating text until it naturally decides its answer is complete.
* `-ngl 99`: Stands for "Number of GPU Layers". By setting this to a high number like 99, we force the engine to offload the entire model to the GPU for maximum speed.
* `-c 35000`: Sets the context window (how much text the AI can remember at once).

If you want to run this model at a context >35k on this GPU, remove the -ngl 99 flag.

In [ ]:
# Run the compiled CLI application to talk to the model
!./build/bin/llama-cli -m gemma-model.gguf -p "Explain the concept of open source software to a 10 year old." -n 24000 -ngl 99 -c 35000

## Step 5: Stop the Model & Clear GPU Memory
If you want to change your prompt, adjust the run flags, or load a completely different model, you first need to stop the current AI and free up the GPU's memory (VRAM).

Simply clicking the "Stop" button on the cell above doesn't always work perfectly. Sometimes, Colab leaves the process running in the background, which will cause an "Out of Memory" error the next time you try to run the engine.

First click the "Stop" button on the cell above and then, run the cell below to explicitly force-quit `llama.cpp` and wipe the GPU memory clean so you can start fresh.

In [ ]:
# 1. Force quit any running llama-cli processes
!pkill -9 -f llama-cli

# 2. Check the GPU to confirm the memory usage has dropped back to ~0MB
!nvidia-smi

Thu Jul 16 03:49:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!./build/bin/llama-cli -m gemma-model.gguf -cnv

/bin/bash: line 1: ./build/bin/llama-cli: No such file or directory


In [ ]:
!pwd

/content


In [ ]:
!./llama.cpp/build/bin/llama-cli -m llama.cpp/gemma-model.gguf -cnv

/bin/bash: line 1: ./llama.cpp/build/bin/llama-cli: No such file or directory


[Errno 2] No such file or directory: 'llama.cpp'
/content


In [ ]:
dir

<function dir>